# Data Collection

Builds the index of RBI Monetary Policy Committee documents (Resolutions, Minutes, Governor's Statements) since October 2016, downloads the source HTML/PDF for each, and pulls USD/INR daily data via `yfinance`.

RBI's policy archive does not expose a stable public API. The year-selection control on the archive page is a custom JavaScript handler (`GetYear()`) that sets a hidden form field (`hdnYear`) and submits a plain POST request, rather than the standard ASP.NET `__doPostBack` mechanism. This was identified by inspecting live network traffic and is replicated directly in `src/scraper.py`.

## Step 0 — Resolve the `src/` module path

In [ ]:
import sys
from pathlib import Path

# This notebook lives in notebooks/, but the code we need lives in ../src
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Project root:", PROJECT_ROOT)
print("Is this right? It should be the rbi-sentiment-market-forecast folder, NOT notebooks.")

Confirms `PROJECT_ROOT` resolves to the repository root rather than `notebooks/`, which the import in the next cell depends on.

## Step 1 — Build the document index

Iterates every financial year from 2016-17 onward, replaying the `hdnYear` POST for each, and parses the resulting document listing into a structured index (date, title, document type, source URL).

In [ ]:
from scraper import scrape_index

years = [f"{y}-{y+1}" for y in range(2016, 2027)]
doc_index = scrape_index(years)

print(f"\nFound {len(doc_index)} documents total")

Expected distribution: roughly 55-65 each of `resolution`, `minutes`, and `governor_statement`, classified from document titles (see `classify()` in `src/scraper.py`, which handles both the modern "Governor's Statement" title format and the pre-2020 "Statement by [Governor's Name], Governor..." format).

In [ ]:
doc_index["doc_type"].value_counts()

In [ ]:
doc_index.head(10)

A `doc_type` distribution dominated by `other` indicates the page structure has changed and the classifier in `src/scraper.py` needs updating.

Persist the index to disk:

In [ ]:
from paths import DOC_INDEX_CSV
doc_index.to_csv(DOC_INDEX_CSV, index=False)
print("Saved to:", DOC_INDEX_CSV)

## Step 2 — Download source documents

Downloads HTML and, where available, PDF for every Resolution, Governor's Statement, and Minutes in the index. PDF is used as the primary text-extraction source in the cleaning stage (Notebook 02); HTML is retained as a fallback for documents where the downloaded PDF does not match the indexed document (see `KNOWN_BAD_PDF_PRIDS` in `src/utils.py`).

In [ ]:
from download import main as download_all
download_all()

Verify HTML/PDF counts against the manifest:

In [ ]:
import pandas as pd
from paths import DOWNLOAD_MANIFEST_CSV, HTML_DIR, PDF_DIR

manifest = pd.read_csv(DOWNLOAD_MANIFEST_CSV)
print(f"HTML files: {len(list(HTML_DIR.glob('*.html')))}")
print(f"PDF files: {len(list(PDF_DIR.glob('*.pdf')))}")
manifest.head()

## Step 3 — USD/INR daily series

In [ ]:
from fetch_market_data import main as fetch_fx
fetch_fx()

## Step 4 — Manual data acquisition

Two market series are not available via a free API and are acquired manually:

- **10-year G-Sec daily yield** — downloaded from investing.com (https://in.investing.com/rates-bonds/india-10-year-bond-yield-historical-data), saved to `data/market/india_10y_gsec_raw.csv`.
- **CPI surprise** (actual inflation vs. RBI's stated forecast) — not used in the current analysis; would require hand-compiling MOSPI actuals against each Resolution's stated CPI projection.